# Задача 05. RFM-сегментация клиентов

**Постановка задачи:** нужно разделить клиентов по RFM-подходу:

- `R` — насколько давно был последний заказ;
- `F` — сколько заказов сделал клиент;
- `M` — сколько денег принёс клиент.

Для простоты сделать 3 уровня по каждому признаку и собрать итоговый сегмент: `champions`, `loyal`, `at_risk`, `sleeping`, `regular`. В расчёт брать доставленные заказы.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path('../data/marketplace.sqlite')
conn = sqlite3.connect(DB_PATH)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print(f'База подключена: {DB_PATH.resolve()}')

In [ ]:
query = r"""
WITH order_finance AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_date,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_pct / 100.0)) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'delivered'
    GROUP BY o.order_id, o.customer_id, o.order_date
), base AS (
    SELECT
        customer_id,
        CAST(julianday((SELECT DATE(MAX(order_date), '+1 day') FROM order_finance)) - julianday(MAX(order_date)) AS INTEGER) AS recency_days,
        COUNT(order_id) AS frequency,
        SUM(revenue) AS monetary
    FROM order_finance
    GROUP BY customer_id
), scored AS (
    SELECT
        *,
        CASE
            WHEN recency_days <= 60 THEN 3
            WHEN recency_days <= 180 THEN 2
            ELSE 1
        END AS r_score,
        CASE
            WHEN frequency >= 4 THEN 3
            WHEN frequency >= 2 THEN 2
            ELSE 1
        END AS f_score,
        CASE
            WHEN monetary >= 120000 THEN 3
            WHEN monetary >= 40000 THEN 2
            ELSE 1
        END AS m_score
    FROM base
), segmented AS (
    SELECT
        *,
        CASE
            WHEN r_score = 3 AND f_score = 3 AND m_score >= 2 THEN 'champions'
            WHEN r_score >= 2 AND f_score >= 2 THEN 'loyal'
            WHEN r_score = 1 AND (f_score >= 2 OR m_score >= 2) THEN 'at_risk'
            WHEN r_score = 1 AND f_score = 1 THEN 'sleeping'
            ELSE 'regular'
        END AS rfm_segment
    FROM scored
)
SELECT
    rfm_segment,
    COUNT(customer_id) AS customers,
    ROUND(AVG(recency_days), 1) AS avg_recency_days,
    ROUND(AVG(frequency), 2) AS avg_frequency,
    ROUND(AVG(monetary), 2) AS avg_monetary,
    ROUND(SUM(monetary), 2) AS total_revenue
FROM segmented
GROUP BY rfm_segment
ORDER BY total_revenue DESC;
"""

df = pd.read_sql_query(query, conn)
df